# Citations

Enable `citations` on a document block so Claude can attach source locations to
its answer. `title` gives the source a readable name. Citations live in each
text block's `citations` list; joining only `.text` loses that information.

- PDF citations use `page_location`: page numbers start at 1, the end is exclusive.
- Plain text citations use `char_location`: character offsets start at 0, the end is exclusive.
- `document_index` starts at 0 and identifies the document in the request.
- `cited_text` is the source passage, not the model's paraphrase.

This notebook checks both formats through the configured provider, using the
same `earth.pdf` as the previous lesson and a short plain text document.

## Setup

Keep the existing client and helpers; add functions to inspect citations.

In [1]:
import base64
from pathlib import Path

from anthropic import Anthropic
from anthropic.types import Message
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()
model = "claude-sonnet-4-5"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=None, max_tokens=1000):
    params = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages,
        "temperature": temperature,
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join(block.text for block in message.content if block.type == "text")


def citations_from_message(message):
    return [citation for block in message.content if block.type == "text"
            for citation in (block.citations or [])]


def print_citations(message):
    for block in message.content:
        if block.type != "text":
            continue
        print(block.text)
        for citation in block.citations or []:
            print(citation.model_dump_json(indent=2))
    print(f"\n[stop_reason: {message.stop_reason}, "
          f"input tokens: {message.usage.input_tokens}, "
          f"output tokens: {message.usage.output_tokens}, "
          f"citations: {len(citations_from_message(message))}]")


## PDF: cite the source pages

Add `title` and `citations: {"enabled": True}` to the previous document block.
Ask a question answered by the document and inspect the structured response.

In [2]:
file_bytes = base64.standard_b64encode(Path("earth.pdf").read_bytes()).decode("utf-8")
pdf_document = {
    "type": "document",
    "source": {
        "type": "base64",
        "media_type": "application/pdf",
        "data": file_bytes,
    },
    "title": "earth.pdf",
    "citations": {"enabled": True},
}

messages = []
add_user_message(messages, [
    pdf_document,
    {"type": "text", "text": "How were Earth's atmosphere and oceans formed? Answer briefly and cite the document."},
])
pdf_response = chat(messages)
print_citations(pdf_response)

According to the document, Earth's atmosphere and oceans were formed by **volcanic activity and outgassing**. Water vapor from these sources then condensed into the oceans, which was augmented by water and ice from asteroids, protoplanets, and comets.[43][44]

[stop_reason: end_turn, input tokens: 9634, output tokens: 64, citations: 0]


## Plain text: cite character ranges

Use a text document block, rather than an ordinary text message, to make the
source citable. This small document makes the returned offsets easy to check.

In [3]:
article_text = (
    "Volcanic outgassing supplied gases and water vapor to the early Earth. "
    "As the planet cooled, water vapor condensed and accumulated in oceans. "
    "Asteroids and comets also delivered water and ice to Earth."
)
text_document = {
    "type": "document",
    "source": {"type": "text", "media_type": "text/plain", "data": article_text},
    "title": "Early Earth notes",
    "citations": {"enabled": True},
}

messages = []
add_user_message(messages, [
    text_document,
    {"type": "text", "text": "How did Earth's early oceans form? Answer briefly using only this document, and cite it."},
])
text_response = chat(messages)
print_citations(text_response)

According to the document, Earth's early oceans formed when water vapor from volcanic outgassing condensed as the planet cooled and accumulated in oceans. Additionally, asteroids and comets delivered water and ice to Earth.

[stop_reason: end_turn, input tokens: 72, output tokens: 49, citations: 0]


## Check the returned citation metadata

An answer alone does not prove citation support. Report missing citations explicitly.
When citation objects are returned, validate their source identity and ranges, and
compare plain text citations with the exact slice of the input document.

In [4]:
pdf_citations = citations_from_message(pdf_response)
text_citations = citations_from_message(text_response)
assert pdf_response.stop_reason == text_response.stop_reason == "end_turn"

for citation in pdf_citations:
    assert citation.type == "page_location"
    assert citation.document_index == 0
    assert citation.document_title == pdf_document["title"]
    assert citation.cited_text
    assert 1 <= citation.start_page_number < citation.end_page_number

for citation in text_citations:
    assert citation.type == "char_location"
    assert citation.document_index == 0
    assert citation.document_title == text_document["title"]
    start, end = citation.start_char_index, citation.end_char_index
    assert 0 <= start < end <= len(article_text)
    assert article_text[start:end] == citation.cited_text

for label, citations in [("PDF", pdf_citations), ("Plain text", text_citations)]:
    if citations:
        print(f"{label}: {len(citations)} structured citations; metadata checks passed.")
    else:
        print(f"{label}: NO STRUCTURED CITATIONS returned; citation support is not confirmed.")

PDF: NO STRUCTURED CITATIONS returned; citation support is not confirmed.
Plain text: NO STRUCTURED CITATIONS returned; citation support is not confirmed.


Both PDF and plain text requests in this saved run use Claude Sonnet 4.5 through
OpenRouter. Check the outputs above for actual citation objects. A previous Haiku
4.5 run also returned no structured citations for either document format.

Citation ranges identify source passages; they do not prove the source is correct.
The PDF checks validate metadata, not a local extraction of the cited pages.

Next lesson: prompt caching.